# CARA v6.1 — Kaggle, two GPUs

**Default: 1 pair · K=5 source + 5 target captions · 300 updates.**
Fork of v6: same FINER/Grad-ECLIP sources, loss, FP32 CLIP features and crop schedule.
Steps 1–150: area .3/.4/.5/.6/.7; then .76/.82/.88/.94/1.0. Random location from update 150.

**Kaggle setup:** attach your dataset, select **GPU T4 × 2**, enable Internet for the
initial dependency/model downloads. Run all from a fresh kernel. Bootstrap only installs
missing/mismatched packages; it leaves Kaggle's CUDA torch/torchvision unchanged. If a package
being replaced was already imported, restart the kernel once and Run all again.

Set `NUM_SAMPLES=200` or `1000` later. `RUN_MODE` only labels artifacts.
Set `XAI_METHOD="grad_eclip"` or `"qwen_attention"`.
Results go to `/kaggle/working/kltn_attack_v6_1_cara_kaggle`; keep this directory for resume.
To review pairing before model loading, stop after the pair-preview cell before running the next cell.

**Two-GPU execution:** G14 stays on GPU1; B16+B32 and Qwen/XAI stay on GPU0.
Two persistent workers launch CLIP branches concurrently and preserve the pixel autograd graph.
Only a 224×224 crop and small feature tensors cross devices. This is model parallelism;
a second copy of G14 is not loaded. Utilization can differ because G14 is the largest branch.

**Verification scope:** CPU/tiny-model equivalence checks are provided. No real Kaggle
two-GPU pretrained benchmark or measured speedup is claimed. At runtime, a gradient probe,
per-sample elapsed time and both GPUs' memory profiles are written to artifacts.

In [ ]:
NOTEBOOK_CODE_SHA256 = '5cae0b2342dde3859a0b525fc80279818f54353a8eb37f970f30ba5f84f52b54'

## Thiết kế thí nghiệm và nguồn

**Ngân hàng:** Qwen3-VL caption cùng ảnh K lần bằng sampling, bắt buộc K caption khác nhau.
Positive = caption target; negative = caption source tương ứng. Không dùng negative-object creator
của FINER để bịa thêm negative. Caption instruction là thiết kế CARA, không phải prompt từ paper.

**Object selection:** giữ nguyên `SYSTEM_MSG_OBJECTS`, `FEW_SHOT_OBJECTS`, `_messages_objects`
và `_split_objects_phrase` từ [FINER](https://github.com/ExplainableML/finer/tree/d67831696e122d01c9823114835f7f50fa07b44c/train/generate_training_data).
Đổi backend inference từ vLLM sang Qwen3-VL/HF; giữ nguyên nội dung messages, kể cả few-shot.
Caption ngắn hơn 512 tokens được kiểm tra; không âm thầm truncate hay thêm concepts.
Ưu tiên các object phrases xuất hiện trong nhiều caption; lưu tất cả output gốc để review
(parser không thể chứng minh mọi object là grounded).

**Grad-ECLIP:** lấy **nguyên văn** 4 hàm `attention_layer`, `clip_encode_dense`, `sim_qk`,
`grad_eclip` từ [author notebook](https://github.com/Cyang-Zhao/Grad-Eclip/blob/e370e6cb194faf2020f5d1ed268f9d57e91a38e6/grad_eclip_image.ipynb),
`n=1`, OpenAI CLIP ViT-B/16 FP16. Giữ cách tác giả tái dựng single-head attention cuối,
gradient tại CLS attention output, QK cosine min-max và ReLU. Đây là Grad-ECLIP author forward;
không thay bằng gradient × input hoặc Grad-CAM. XAI model này tách khỏi 3 surrogate encoders.
Code tác giả có thể sinh NaN khi QK range=0: notebook dừng hoặc loại map lỗi có log, không sửa công thức.

**Qwen attention:** dùng attention weights từ **official eager forward**, không xấp xỉ bằng
dot product thiếu RoPE/causal mask: [Transformers v4.57.1 implementation](https://github.com/huggingface/transformers/blob/v4.57.1/src/transformers/models/qwen3_vl/modeling_qwen3_vl.py).
Chọn token rows ứng với concept trong user query, image-token keys từ config, reshape theo
`image_grid_thw / spatial_merge_size`. Softmax của model tính trên toàn bộ causal keys.
Mean heads, concept tokens và layer được chọn là **quy ước CARA**; không gọi đây là một phương pháp
XAI được Qwen authors đề xuất, cũng không coi attention là giải thích nhân quả.
[Qwen3-VL official inference](https://github.com/QwenLM/Qwen3-VL/tree/96588727e44c78b25ba03ea03b8e12f7e64fd0da).

**CLIP preprocessing:** resize tensor bicubic 224×224, clamp [0,1], mean/std CLIP giống
[MPC author adapters](https://github.com/LiYuanBoJNU/MPCAttack/tree/7b195a5bcbaeaf1e8534d101199810a2782caad3/surrogates/FeatureExtractors).
Không tuyên bố tái lập MPC loss/optimizer. Normalize mỗi modality, weighted fusion và lựa chọn
nearest-mean bên dưới là thiết kế của Hypothesis 2, không phải công thức lấy từ MPC.
Không có FOA, diffusion, k-means hoặc loss phụ.

In [ ]:
# Kaggle configuration — select T4 x 2, then Run all.
from pathlib import Path
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
INSTALL_DEPENDENCIES = True   # smart bootstrap; no reinstall when pinned packages match
RUN_EXPERIMENT = True
RUN_MODE = "pilot"                 # artifact label: "pilot" | "full"; count uses NUM_SAMPLES
NUM_SAMPLES = 1                     # number of pairs: 1 for test; set 200 or 1000 later
XAI_METHOD = "grad_eclip"           # "grad_eclip" | "qwen_attention"
DATASET_ROOT = Path('/kaggle/input')  # discover exactly one bigscale/target_images across mounts
SOURCE_DIR = None                    # auto-discover exactly one "bigscale" directory
TARGET_DIR = None                    # auto-discover exactly one "target_images" directory
PAIR_MANIFEST = None                 # JSONL: {"index": 0, "source": "...", "target": "..."}
OUTPUT_ROOT = Path('/kaggle/working/kltn_attack_v6_1_cara_kaggle')
SAMPLE_START = 0
SEED = 2026
K = 5
STEPS = 300
REFRESH_EVERY = 30
RANDOM_FROM_STEP = 150               # 1-based: updates 150..300 random, XAI still measured
EPSILON_255 = 16                     # CARA default, not specified by hypothesis
ALPHA_255 = 1                        # sign gradient descent; no random start / momentum
TEMPERATURE = 0.07
BACKBONES = ('G14', 'B16', 'B32')
CLIP_IDS = {'G14': 'laion/CLIP-ViT-g-14-laion2B-s12B-b42K',
            'B16': 'openai/clip-vit-base-patch16', 'B32': 'openai/clip-vit-base-patch32'}
# Required: two CUDA GPUs. Persistent parallel workers: G14 on GPU1, B16+B32 on GPU0.
CLIP_DEVICE_MAP = {'G14': 'cuda:1', 'B16': 'cuda:0', 'B32': 'cuda:0'}
ATTACK_DEVICE = 'cuda:0'
QWEN_DEVICE = 'cuda:0'
QWEN_ID = 'Qwen/Qwen3-VL-2B-Instruct'
EVAL_MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'  # separate caption VLM, configurable
EVAL_DEVICE = 'cuda:0'
EVALUATE_CAPTIONS = True
MODEL_REVISION_OVERRIDES = {}          # repo_id -> immutable commit; first run resolves once
IMAGE_WEIGHTS = {'G14': 0.6, 'B16': 0.6, 'B32': 0.6}
TEXT_WEIGHTS = {'G14': 0.4, 'B16': 0.4, 'B32': 0.4}
CAPTION_TEMPERATURE = 0.8
CAPTION_TOP_P = 0.9
CAPTION_MAX_NEW_TOKENS = 96
MAX_BANK_ATTEMPTS = 40
MAX_CONCEPTS = 3
QWEN_ATTENTION_LAYER = -1             # -1 = last decoder layer; saved as resolved index
QWEN_MAX_PIXELS = 256 * 32 * 32       # bounds attention memory; no video / multi-image input
XAI_MAX_SIDE = 448
SAVED_TENSORS_ON_CPU = False          # avoid copying all saved weights/activations over PCIe
OFFLOAD_BACKBONES = ()                # optional memory tradeoff, e.g. ("G14",); slower
CLIP_ATTN_IMPLEMENTATION = "sdpa"    # official HF CLIP backend; FP32, no AMP
CPU_THREADS = 2
EVAL_TEXT_BATCH_PAIRS = 4             # batch 4*(2+2*K) captions, both GPU groups
RUN_TWO_GPU_SMOKE_TEST = True
LOG_EVERY = 30

In [ ]:
# Smart bootstrap; do not pip-upgrade torch/torchvision on Kaggle.
if INSTALL_DEPENDENCIES:
    import importlib, importlib.util, json, subprocess, sys
    from importlib import metadata as setup_metadata
    pinned = {'transformers': '4.57.1', 'accelerate': '1.11.0',
              'huggingface_hub': '0.36.0', 'tokenizers': '0.22.1', 'ftfy': '6.3.1'}
    requirements = []
    changed_modules = []
    for name, version in pinned.items():
        try: current = setup_metadata.version(name)
        except setup_metadata.PackageNotFoundError: current = None
        if current != version:
            requirements.append(f'{name}=={version}')
            changed_modules.append(name)
    for name in ('regex', 'Pillow', 'numpy', 'pandas', 'matplotlib', 'nbformat'):
        try: setup_metadata.version(name)
        except setup_metadata.PackageNotFoundError: requirements.append(name)
    clip_commit = 'd05afc436d78f1c48dc0dbf8e5980a9d471f35f6'
    try:
        direct_url = json.loads(setup_metadata.distribution('clip').read_text('direct_url.json') or '{}')
        clip_pinned = direct_url.get('vcs_info', {}).get('commit_id') == clip_commit
    except (setup_metadata.PackageNotFoundError, ValueError): clip_pinned = False
    if not clip_pinned:
        requirements.append('git+https://github.com/openai/CLIP.git@' + clip_commit)
        changed_modules.append('clip')
    if requirements:
        imported = [name for name in changed_modules if name in sys.modules]
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
            '--disable-pip-version-check', *requirements], check=True)
        importlib.invalidate_caches()
        if imported:
            raise SystemExit('Updated already-imported packages: ' + ', '.join(imported) +
                             '. Restart kernel and Run all again.')
    else: print('Pinned runtime already installed; bootstrap skipped.')

In [ ]:
import ast, contextlib, gc, hashlib, io, json, math, random, re, sys, time
from collections import Counter
from importlib import metadata as package_metadata
from typing import List
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision import datasets
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, InterpolationMode

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temp.replace(path)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def free_cuda():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def image_tensor(image):
    return torch.from_numpy(np.array(image, dtype=np.uint8, copy=True)).permute(2,0,1).unsqueeze(0).float()/255

def tensor_image(x):
    a = x.detach().cpu().squeeze(0).clamp(0,1).mul(255).round().byte().permute(1,2,0).numpy()
    return Image.fromarray(a)

def open_rgb(path):
    with Image.open(path) as im: return ImageOps.exif_transpose(im).convert('RGB')

def reduced_image(image, side=XAI_MAX_SIDE):
    image = image.copy(); image.thumbnail((side, side), Image.Resampling.BICUBIC)
    return image

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

from concurrent.futures import ThreadPoolExecutor, wait
torch.set_num_threads(CPU_THREADS)

In [ ]:
def gpu_profile(synchronize=True):
    if not torch.cuda.is_available(): return []
    records=[]
    for i in range(min(torch.cuda.device_count(),2)):
        if synchronize: torch.cuda.synchronize(i)
        free,total=torch.cuda.mem_get_info(i)
        records.append({'device':f'cuda:{i}','name':torch.cuda.get_device_name(i),
            'allocated_gib':torch.cuda.memory_allocated(i)/2**30,
            'reserved_gib':torch.cuda.memory_reserved(i)/2**30,
            'peak_allocated_gib':torch.cuda.max_memory_allocated(i)/2**30,
            'free_gib':free/2**30,'total_gib':total/2**30})
    return records

def kaggle_preflight():
    if not torch.cuda.is_available() or torch.cuda.device_count()<2:
        raise RuntimeError('Select Kaggle Settings -> Accelerator -> GPU T4 x 2, then restart kernel.')
    expected={'G14':'cuda:1','B16':'cuda:0','B32':'cuda:0'}
    if CLIP_DEVICE_MAP!=expected or ATTACK_DEVICE!='cuda:0' or QWEN_DEVICE!='cuda:0':
        raise ValueError('v6.1 uses G14 on cuda:1 and B16/B32/Qwen on cuda:0.')
    if not isinstance(SAVED_TENSORS_ON_CPU,bool) or set(OFFLOAD_BACKBONES)-set(BACKBONES):
        raise ValueError('Invalid activation-offload settings.')
    if type(EVAL_TEXT_BATCH_PAIRS) is not int or EVAL_TEXT_BATCH_PAIRS<1:
        raise ValueError('EVAL_TEXT_BATCH_PAIRS must be a positive integer.')
    print('Two-GPU preflight:',json.dumps(gpu_profile(),indent=2))

def two_gpu_gradient_probe(ensemble,source):
    image=source.detach().to(ensemble.device).requires_grad_(True)
    started=time.perf_counter()
    features=ensemble.images(image)
    if any(not feature.requires_grad for feature in features):
        raise RuntimeError('A CLIP worker detached its pixel-gradient graph.')
    q=unit(torch.cat(features,dim=-1))
    weights=torch.linspace(.2,1.,q.shape[-1],device=q.device)[None,:]
    gradient,=torch.autograd.grad((q*weights).sum(),image)
    if not torch.isfinite(gradient).all() or gradient.abs().sum()==0:
        raise RuntimeError('Cross-device pixel-gradient probe failed.')
    devices={name:str(next(model.parameters()).device) for name,model in ensemble.models.items()}
    if set(devices.values())!={'cuda:0','cuda:1'}:
        raise RuntimeError(f'CLIPs did not occupy both GPUs: {devices}')
    report={'status':'passed','model_devices':devices,'query_shape':list(q.shape),
        'feature_shapes':[list(f.shape) for f in features],
        'gradient_l1':float(gradient.abs().sum()),'gpus':gpu_profile(),
        'seconds':time.perf_counter()-started}
    del q,gradient,image,features
    return report

## Công thức và lịch crop

Với model $m$, $u_m=\operatorname{norm}(E^I_m(I))$, $v_{mj}=\operatorname{norm}(E^T_m(c_j))$.
Mỗi caption dùng cùng ảnh thuộc phía tương ứng:
$b_{mj}=\operatorname{norm}(w^I_m u_m+w^T_m v_{mj})$,
$z_j=\operatorname{norm}([b_{G,j};b_{B16,j};b_{B32,j}])$.
Chọn **cùng một index** $j^*=\arg\min_j\|z^t_j-\frac1K\sum_i z^t_i\|_2$ cho cả ensemble,
không lấy mean thay cho một caption thật. Các hệ số fusion áp dụng cho anchor; query là
$q=\operatorname{norm}([\operatorname{norm}(E^I_G(I_{adv}^{crop}));\ldots])$.
Query chỉ có image vì caption generation không khả vi; không ghép caption adversarial cố định vào query.

$$\mathcal L=-\log\frac{\exp(q^\top z^t_{j^*}/\tau)}{\exp(q^\top z^t_{j^*}/\tau)+\sum_{j=1}^K\exp(q^\top z^s_j/\tau)}.$$

Gradient **descent** trên loss này kéo target và đẩy K source. Do source captions có thể gần nghĩa,
K negatives không bảo đảm độc lập; kiểm tra pairwise cosine trong artifact anchors.

| Update (1-based) | Area fraction | Location |
|---|---:|---|
| 1–30 / 31–60 / 61–90 / 91–120 | .3 / .4 / .5 / .6 | maximum XAI mass window |
| 121–149 | .7 | maximum XAI mass window |
| 150 | .7 | random |
| 151–180 / 181–210 / 211–240 / 241–270 | .76 / .82 / .88 / .94 | random |
| 271–300 | 1.0 | full original frame |

Map được đo trên ảnh hiện tại ở step **0,30,...,300**. Map ở 30 dùng cho updates 31–60;
sau update 150 vẫn đo XAI nhưng không dùng map để chọn crop. Crop giữ tỉ lệ ảnh gốc:
height/width ≈ sqrt(area) lần height/width ban đầu, có làm tròn pixel. Random chọn vị trí
đều trong canvas với kích thước này; không dùng RandomResizedCrop làm thay đổi area ngẫu nhiên.
Loss chỉ nhìn crop; nhiễu chỉ được cập nhật trong box. Ảnh lưu giữ nguyên kích thước source gốc.

In [ ]:
def unit(x):
    x = x.float()
    if not torch.isfinite(x).all() or (x.norm(dim=-1) < 1e-12).any():
        raise ValueError('Nonfinite or zero feature vector.')
    return F.normalize(x, dim=-1)

def fuse_features(images, texts, image_weights, text_weights):
    if not len(images) == len(texts) == len(image_weights) == len(text_weights):
        raise ValueError('Backbone fusion lengths differ.')
    blocks = []
    for im, tx, wi, wt in zip(images, texts, image_weights, text_weights):
        if not math.isfinite(wi+wt) or min(wi, wt) < 0 or wi+wt <= 0:
            raise ValueError('Fusion weights must be finite, nonnegative, not both zero.')
        blocks.append(unit(wi*unit(im) + wt*unit(tx)))
    return unit(torch.cat(blocks, dim=-1))

def nearest_mean_index(bank):
    if bank.ndim != 2 or len(bank) < 1: raise ValueError('Expected [K,D] bank.')
    return int(((bank-bank.mean(0, keepdim=True))**2).sum(-1).argmin())

def infonce(query, positive, negatives, temperature):
    if not 0 < temperature < float('inf'): raise ValueError('Invalid temperature.')
    q, p, n = unit(query), unit(positive), unit(negatives)
    if q.ndim != 2 or p.shape != q.shape or n.ndim != 2 or n.shape[1] != q.shape[1] or len(n) < 1:
        raise ValueError('Expected query/positive [B,D], negatives [K,D].')
    logits = torch.cat([(q*p).sum(-1, keepdim=True), q@n.T], dim=1)/temperature
    return F.cross_entropy(logits, torch.zeros(len(q), dtype=torch.long, device=q.device))

def crop_area(step, steps=300, refresh_every=30):
    # Area grows 0.3 -> 0.7 during updates 1..150, then evenly to 1.0.
    areas = (0.3, 0.4, 0.5, 0.6, 0.7, 0.76, 0.82, 0.88, 0.94, 1.0)
    if (steps, refresh_every) != (300, 30) or type(step) is not int or not 1 <= step <= steps:
        raise ValueError('Expected 300 updates, 30-update stages, and an integer step in [1,300].')
    return areas[(step-1)//refresh_every]

def crop_mode(step, random_from=150):
    return 'random' if step >= random_from else 'xai'

def select_box(heatmap, area, mode, generator):
    heat = torch.as_tensor(heatmap).detach().float().cpu()
    if heat.ndim != 2 or not torch.isfinite(heat).all() or not 0 < area <= 1:
        raise ValueError('Invalid spatial map/area.')
    H, W = heat.shape
    h = max(1, min(H, int(round(H*math.sqrt(area)))))
    w = max(1, min(W, int(round(W*math.sqrt(area)))))
    if area == 1: return (0, 0, H, W)
    if mode == 'random':
        y = int(torch.randint(H-h+1, (1,), generator=generator))
        x = int(torch.randint(W-w+1, (1,), generator=generator))
    elif mode == 'xai':
        # Integral-image exact exhaustive fixed-size window sum, O(HW), no approximate grid.
        sat = F.pad(heat.double(), (1,0,1,0)).cumsum(0).cumsum(1)
        sums = sat[h:,w:]-sat[:-h,w:]-sat[h:,:-w]+sat[:-h,:-w]
        index = int(sums.reshape(-1).argmax()); y, x = divmod(index, W-w+1)
    else: raise ValueError('Unknown crop location mode.')
    return y, x, h, w

def masked_pgd_step(adv, original, gradient, box, alpha, epsilon):
    if not torch.isfinite(gradient).all(): raise FloatingPointError('Nonfinite attack gradient.')
    y, x, h, w = box
    out = adv.detach().clone()
    out[:,:,y:y+h,x:x+w] -= alpha*gradient[:,:,y:y+h,x:x+w].sign()
    return torch.maximum(torch.minimum(out, original+epsilon), original-epsilon).clamp(0,1).detach()

def dispersion(heatmap):
    # Same 32x32 normalized coordinates for comparable entropy across patch grids/resolutions.
    heat = torch.as_tensor(heatmap).float().detach().cpu()
    if heat.ndim != 2 or not torch.isfinite(heat).all() or (heat < 0).any() or heat.sum() <= 0:
        raise ValueError('Undefined dispersion: map is nonfinite/negative/zero.')
    p = F.interpolate(heat[None,None], size=(32,32), mode='bilinear', align_corners=False)[0,0]
    p = p/p.sum(); nz = p[p>0]
    entropy = -(nz*nz.log()).sum()
    yy, xx = torch.meshgrid(torch.linspace(0,1,32), torch.linspace(0,1,32), indexing='ij')
    cy, cx = (p*yy).sum(), (p*xx).sum()
    return {'normalized_entropy': float(entropy/math.log(p.numel())),
            'effective_support': float(torch.exp(entropy)/p.numel()),
            'spatial_variance': float((p*((yy-cy)**2+(xx-cx)**2)).sum()),
            'centroid_y': float(cy), 'centroid_x': float(cx)}

In [ ]:
def sample_count():
    if type(NUM_SAMPLES) is not int or NUM_SAMPLES < 1:
        raise ValueError('NUM_SAMPLES must be a positive integer (e.g. 1, 200, 1000).')
    return NUM_SAMPLES

def prepare_pairs():
    n = sample_count()
    def find_dir(label, override):
        if override is not None:
            p = Path(override)
            if not p.is_dir(): raise FileNotFoundError(p)
            return p
        if not DATASET_ROOT.is_dir(): raise FileNotFoundError(f'Edit DATASET_ROOT: {DATASET_ROOT}')
        found = ([DATASET_ROOT] if DATASET_ROOT.name == label else []) + list(DATASET_ROOT.rglob(label))
        found = [p for p in found if p.is_dir()]
        if len(found) != 1: raise ValueError(f'Expected exactly one {label}: {found}. Set explicit directory.')
        return found[0]
    if PAIR_MANIFEST is not None:
        path = Path(PAIR_MANIFEST)
        pairs = [json.loads(s) for s in path.read_text(encoding='utf-8').splitlines() if s.strip()]
        for row in pairs:
            for role in ('source', 'target'):
                p = Path(row[role]); row[role] = str((p if p.is_absolute() else path.parent/p).resolve())
    else:
        # Preserve ImageFolder sorted class/file pairing from the existing attack notebooks.
        s = datasets.ImageFolder(find_dir('bigscale', SOURCE_DIR))
        t = datasets.ImageFolder(find_dir('target_images', TARGET_DIR))
        if len(s) != len(t): raise ValueError('Unequal source/target count; provide an explicit pair manifest.')
        pairs = [{'index': i, 'source': a[0], 'target': b[0]} for i,(a,b) in enumerate(zip(s.samples,t.samples))]
    pairs = pairs[SAMPLE_START:SAMPLE_START+n]
    if len(pairs) != n: raise ValueError(f'Need {n} pairs, found {len(pairs)}.')
    indices = [p['index'] for p in pairs]
    if any(type(i) is not int or i < 0 for i in indices) or len(set(indices)) != n:
        raise ValueError('Pair indices must be unique nonnegative integers.')
    for row in pairs:
        if Path(row['source']).resolve() == Path(row['target']).resolve(): raise ValueError('Identical source/target path.')
        row['input_sha256'] = {r: file_hash(row[r]) for r in ('source','target')}
        if row['input_sha256']['source'] == row['input_sha256']['target']:
            raise ValueError('Identical source and target image bytes.')
    return pairs

def validate_config():
    sample_count()
    kaggle_preflight()
    if RUN_MODE not in {'pilot','full'} or XAI_METHOD not in {'grad_eclip','qwen_attention'}:
        raise ValueError('Choose a known run mode and XAI method.')
    if (STEPS,REFRESH_EVERY,RANDOM_FROM_STEP) != (300,30,150):
        raise ValueError('v6 fixes 300/30/150. Relabel a schedule ablation explicitly.')
    if type(K) is not int or K < 1 or MAX_BANK_ATTEMPTS < K: raise ValueError('Invalid K/attempt budget.')
    if type(SAMPLE_START) is not int or SAMPLE_START < 0: raise ValueError('Invalid SAMPLE_START.')
    if not 0 < TEMPERATURE < float('inf') or not 0 < ALPHA_255 <= EPSILON_255 <= 255:
        raise ValueError('Invalid loss temperature/pixel budget.')
    if EPSILON_255 != int(EPSILON_255): raise ValueError('Use an integer pixel epsilon for exact PNG bounds.')
    if MAX_CONCEPTS < 1 or QWEN_MAX_PIXELS < 32*32 or XAI_MAX_SIDE < 32: raise ValueError('Invalid XAI limits.')
    if not 0 < CAPTION_TEMPERATURE < float('inf') or not 0 < CAPTION_TOP_P <= 1:
        raise ValueError('Invalid caption sampling.')
    if EVALUATE_CAPTIONS and EVAL_MODEL_ID == QWEN_ID: raise ValueError('Use a distinct evaluation model.')
    for mapping in (CLIP_IDS,CLIP_DEVICE_MAP,IMAGE_WEIGHTS,TEXT_WEIGHTS):
        if set(mapping) != set(BACKBONES): raise ValueError('Expected exactly G14/B16/B32 keys.')
    for name in BACKBONES:
        wi,wt = IMAGE_WEIGHTS[name],TEXT_WEIGHTS[name]
        if not math.isfinite(wi+wt) or min(wi,wt)<0 or wi+wt<=0: raise ValueError('Invalid fusion weights.')
    if not torch.cuda.is_available(): raise RuntimeError('Pretrained execution requires CUDA.')
    for d in list(CLIP_DEVICE_MAP.values()) + [ATTACK_DEVICE,QWEN_DEVICE,EVAL_DEVICE]:
        device = torch.device(d)
        if device.type != 'cuda' or (device.index or 0) >= torch.cuda.device_count():
            raise RuntimeError(f'{d} unavailable. Edit CLIP_DEVICE_MAP/devices for this host.')
    if package_metadata.version('transformers') != '4.57.1':
        raise RuntimeError('This adapter is pinned to transformers==4.57.1; run optional setup and restart.')

def resolve_revisions():
    from huggingface_hub import HfApi
    lock_path = OUTPUT_ROOT/'model_revisions.json'
    lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
    ids = list(CLIP_IDS.values()) + [QWEN_ID] + ([EVAL_MODEL_ID] if EVALUATE_CAPTIONS else [])
    for model_id in ids:
        rev = MODEL_REVISION_OVERRIDES.get(model_id, lock.get(model_id))
        if rev is None: rev = HfApi().model_info(model_id).sha
        if not re.fullmatch(r'[a-f0-9]{40}', str(rev)): raise ValueError(f'Expected model commit SHA: {model_id}={rev}')
        lock[model_id] = rev
    atomic_json(lock_path, lock)
    return {name: lock[name] for name in sorted(set(ids))}

In [ ]:
# Immutable embedded author strings; no remote code execution and no source download at runtime.
AUTHOR_SOURCES = {'finer_object_prompts.py': 'SYSTEM_MSG_OBJECTS = (\n    "You are an information extraction assistant.\\n"\n    "From the caption, select up to FIVE main objects. A main object must have at least one descriptive attribute in the caption "\n    "(e.g., color, size, material, possession/with-phrase, appositive, relative clause, or an explicit number).\\n"\n    "Rules:\\n"\n    "• Output only object names that appear in the caption and are part of the described scene. Never invent or infer.\\n"\n    "• Prefer plain object names in the output (omit adjectives), but KEEP explicit numbers/quantifiers if the caption states them "\n    "(e.g., \'two cats\', \'five chickens\', \'a pair of skis\').\\n"\n    "• If multiple mentions share the same name and no explicit number is given, output the plural form (e.g., \'dogs\'). \\n"\n    "• List 1–5 main objects; if only one is present, output just that one.\\n"\n    "Do not add quantifiers like \'some\' unless present in the caption.\\n"\n    "Format:\\n"\n    "Return EXACTLY one line:\\n"\n    "PHRASE=<comma-separated list with \'and\' before the last item, e.g., \'a dog, a cat and two birds\'>\\n"\n    "No trailing period. No extra text."\n)\n\nFEW_SHOT_OBJECTS = [\n    (\n        "A bustling street scene with three blue buses at the curb, two red cars in traffic, "\n        "a tall glass office building behind them, and a bronze statue with a raised arm in the plaza.",\n        "PHRASE=three buses, two cars, an office building and a statue"\n    ),\n    (\n        "On a wooden table, a cast-iron skillet sits beside a blue ceramic vase of sunflowers, "\n        "with red apples and green pears piled high in a stainless steel bowl.",\n        "PHRASE=a skillet, a vase, apples, pears and a bowl"\n    ),\n]\n\nSYSTEM_MSG_NEGOBJ = (\n    "You are a negative object creator.\\n"\n    "You will receive a caption, an object list as PHRASE=..., and REPLACE_INDEX=k (1-based).\\n"\n    "Replace EXACTLY the k-th object in PHRASE with a distinctly different NEGATIVE object.\\n"\n    "\\n"\n    "Keep ALL other objects unchanged and preserve their order and punctuation. Keep the same quantifier/determiner "\n    "for the replaced slot (e.g., \'two cats\' -> \'two bicycles\').\\n"\n    "\\n"\n    "Constraints for the NEGATIVE object:\\n"\n    "• It must be distinctly different from the replaced object (not a synonym; not just singular/plural).\\n"\n    "• It must NOT be a synonym or near-equivalent of ANY object that appears in the caption.\\n"\n    "• It must NOT appear anywhere in the caption (as a whole word, singular or plural).\\n"\n    "• Do not modify any other items; do not reorder items; do not add or remove items.\\n"\n    "\\n"\n    "Edge cases (must follow):\\n"\n    "• If REPLACE_INDEX is greater than the number of objects in PHRASE, replace the LAST object.\\n"\n    "• If REPLACE_INDEX is less than 1, replace the FIRST object.\\n"\n    "\\n"\n    "Self-check (must hold):\\n"\n    "• Same number of items as input; exactly one item (the k-th per the rule above) differs.\\n"\n    "\\n"\n    "Output EXACTLY one line:\\n"\n    "PHRASE=<the new list, same format>\\n"\n    "No extra text. No quotes. No trailing period."\n)\n\n# Few-shots cover different list lengths and positions; each uses REPLACE_INDEX to avoid a last-item bias.\nFEW_SHOT_NEGOBJ = [\n    (\n        # Replace FIRST of three\n        "Caption:\\nA small yellow school bus is parked next to two red cars in front of a brick office building.\\n"\n        "PHRASE=a bus, two cars and an office building\\n"\n        "REPLACE_INDEX=1",\n        "PHRASE=a boat, two cars and an office building"\n    ),\n    (\n        "Caption:\\nTwo hikers stand on a wooden bridge with their dog beside them over a calm river.\\n"\n        "PHRASE=two hikers, a dog, a bridge and a river\\n"\n        "REPLACE_INDEX=2",\n        "PHRASE=two hikers, a bicycle, a bridge and a river"\n    ),\n    (\n        "Caption:\\nOn a side table there is a bowl of fruit, a ceramic vase, and a brass lamp.\\n"\n        "PHRASE=a bowl of fruit, a vase and a lamp\\n"\n        "REPLACE_INDEX=3",\n        "PHRASE=a bowl of fruit, a vase and a clock"\n    ),\n    (\n        "Caption:\\nA dashboard card labeled \'Latency\' shows a graph with a menu icon and a legend for several lines.\\n"\n        "PHRASE=a card, a graph, a menu icon, a legend and lines\\n"\n        "REPLACE_INDEX=3",\n        "PHRASE=a card, a graph, a slider, a legend and lines"\n    ),\n    (\n        "Caption:\\nA booking screen shows a highlighted 12:05–12:30 time slot, a search box, an entry for Animal Kingdom Villas, a restaurant icon, and a market link.\\n"\n        "PHRASE=a time slot, a box, an entry, a restaurant and a market\\n"\n        "REPLACE_INDEX=4",\n        "PHRASE=a time slot, a box, an entry, a ticket counter and a market"\n    ),\n]\n\nCATEGORIES = ["natural_image", "chart_graph", "screenshot_ui", "document_text"]\n\nSYSTEM_MSG_CATEGORY = (\n    "You are a strict classifier. Based ONLY on the caption text, assign EXACTLY ONE label.\\n"\n    "Allowed labels:\\n"\n    "  - natural_image\\n"\n    "  - chart_graph   (includes diagrams, flowcharts, schematics, illustrations)\\n"\n    "  - screenshot_ui (UI/app/website/IDE/terminal screenshots)\\n"\n    "  - document_text (scans, printed pages, receipts, invoices, whiteboards, handwriting)\\n"\n    "\\n"\n    "Rules:\\n"\n    "• Return ONLY the label string, nothing else.\\n"\n    "• If multiple seem plausible, pick the single best one.\\n"\n    "• If unsure, choose natural_image.\\n"\n)\n\nFEW_SHOT_CATEGORY = [\n    ("A line chart comparing quarterly revenue for three products with a legend and axes labels.",\n     "chart_graph"),\n    ("A screenshot of a mobile banking app showing the account balance and a transfer button.",\n     "screenshot_ui"),\n    ("A scanned receipt from a grocery store listing items and totals.",\n     "document_text"),\n    ("A photo of a dog running on a beach at sunset.",\n     "natural_image"),\n    ("A flowchart illustrating an authentication pipeline with decision nodes and arrows.",\n     "chart_graph"),\n]\n\n', 'finer_object_helpers.py': 'def _split_objects_phrase(phrase: str) -> List[str]:\n    """\n    Split \'a dog, a cat and two birds\' -> [\'a dog\',\'a cat\',\'two birds\'].\n    Heuristic: split on the last \' and \' and commas on the left.\n    Used only for lightweight validation; storage remains a single string.\n    """\n    s = (phrase or "").strip()\n    if not s:\n        return []\n    if " and " in s:\n        left, last = s.rsplit(" and ", 1)\n        left_parts = [p.strip() for p in left.split(",") if p.strip()]\n        items = left_parts + [last.strip()]\n    else:\n        items = [p.strip() for p in s.split(",") if p.strip()]\n    return items\n\ndef _messages_objects(caption_text: str) -> List[dict]:\n    """Chat messages for the \'objects\' mode with few-shot."""\n    msgs = [{"role": "system", "content": SYSTEM_MSG_OBJECTS}]\n\n    for cap, ans in FEW_SHOT_OBJECTS:\n        demo_user = (\n            "Caption:\\n" + cap + "\\n\\n"\n            "Task: From this caption, select up to five main objects that have at least one descriptive attribute in the caption. "\n            "Prefer plain object names; keep explicit numbers/quantifiers if present; pluralize repeated names without numbers. "\n            "Return EXACTLY one line:\\nPHRASE=<...>"\n        )\n        msgs.append({"role": "user", "content": demo_user})\n        msgs.append({"role": "assistant", "content": ans})\n\n    task_user = (\n        "Caption:\\n" + caption_text + "\\n\\n"\n        "Task: Select up to five main objects that have at least one descriptive attribute in the caption. "\n        "Prefer plain object names; keep explicit numbers/quantifiers if present; pluralize repeated names without numbers. "\n        "Return EXACTLY one line:\\nPHRASE=<...>"\n    )\n    msgs.append({"role": "user", "content": task_user})\n    return msgs', 'gradeclip_author_functions.py': 'def attention_layer(q, k, v, num_heads=1):\n    "Compute \'Scaled Dot Product Attention\'"\n    tgt_len, bsz, embed_dim = q.shape\n    head_dim = embed_dim // num_heads\n    scaling = float(head_dim) ** -0.5\n    q = q * scaling\n    \n    q = q.contiguous().view(tgt_len, bsz * num_heads, head_dim).transpose(0, 1)\n    k = k.contiguous().view(-1, bsz * num_heads, head_dim).transpose(0, 1)\n    v = v.contiguous().view(-1, bsz * num_heads, head_dim).transpose(0, 1)\n    attn_output_weights = torch.bmm(q, k.transpose(1, 2))\n    attn_output_weights = F.softmax(attn_output_weights, dim=-1)\n    attn_output_heads = torch.bmm(attn_output_weights, v)\n    assert list(attn_output_heads.size()) == [bsz * num_heads, tgt_len, head_dim]\n    attn_output = attn_output_heads.transpose(0, 1).contiguous().view(tgt_len, bsz, embed_dim)\n    attn_output_weights = attn_output_weights.view(bsz, num_heads, tgt_len, -1)\n    attn_output_weights = attn_output_weights.sum(dim=1) / num_heads\n    return attn_output, attn_output_weights\n\ndef clip_encode_dense(x,n):\n    vision_width = clipmodel.visual.transformer.width\n    vision_heads = vision_width // 64\n    print("[vision_width and vision_heads]:", vision_width, vision_heads)\n    \n    # modified from CLIP\n    x = x.half()\n    x = clipmodel.visual.conv1(x)  \n    feah, feaw = x.shape[-2:]\n\n    x = x.reshape(x.shape[0], x.shape[1], -1) \n    x = x.permute(0, 2, 1) \n    class_embedding = clipmodel.visual.class_embedding.to(x.dtype)\n    x = torch.cat([class_embedding + torch.zeros(x.shape[0], 1, x.shape[-1]).to(x), x], dim=1)\n\n    ## scale position embedding as the image w-h ratio\n    pos_embedding = clipmodel.visual.positional_embedding.to(x.dtype)\n    tok_pos, img_pos = pos_embedding[:1, :], pos_embedding[1:, :]\n    pos_h = clip_inres // clip_ksize[0]\n    pos_w = clip_inres // clip_ksize[1]\n    assert img_pos.size(0) == (pos_h * pos_w), f"the size of pos_embedding ({img_pos.size(0)}) does not match resolution shape pos_h ({pos_h}) * pos_w ({pos_w})"\n    img_pos = img_pos.reshape(1, pos_h, pos_w, img_pos.shape[1]).permute(0, 3, 1, 2)\n    print("[POS shape]:", img_pos.shape, (feah, feaw))\n    img_pos = torch.nn.functional.interpolate(img_pos, size=(feah, feaw), mode=\'bicubic\', align_corners=False)\n    img_pos = img_pos.reshape(1, img_pos.shape[1], -1).permute(0, 2, 1)\n    pos_embedding = torch.cat((tok_pos[None, ...], img_pos), dim=1)\n    x = x + pos_embedding\n    x = clipmodel.visual.ln_pre(x)\n    \n    x = x.permute(1, 0, 2)  # NLD -> LND\n    x = torch.nn.Sequential(*clipmodel.visual.transformer.resblocks[:-n])(x)\n\n    attns = []\n    atten_outs = []\n    vs = []\n    qs = []\n    ks = []\n    for TR in clipmodel.visual.transformer.resblocks[-n:]:\n        x_in = x\n        x = TR.ln_1(x_in)\n        linear = torch._C._nn.linear    \n        q, k, v = linear(x, TR.attn.in_proj_weight, TR.attn.in_proj_bias).chunk(3, dim=-1)\n        attn_output, attn = attention_layer(q, k, v, 1)  # vision_heads=1\n        attns.append(attn)\n        atten_outs.append(attn_output)\n        vs.append(v)\n        qs.append(q)\n        ks.append(k)\n        \n        x_after_attn = linear(attn_output, TR.attn.out_proj.weight, TR.attn.out_proj.bias)       \n        x = x_after_attn + x_in\n        x = x + TR.mlp(TR.ln_2(x))\n\n    x = x.permute(1, 0, 2)  # LND -> NLD\n    x = clipmodel.visual.ln_post(x)\n    x = x @ clipmodel.visual.proj\n    return x, x_in, vs, qs, ks, attns, atten_outs, (feah, feaw)\n\ndef sim_qk(q, k):\n    q_cls = F.normalize(q[:1,0,:], dim=-1) \n    k_patch = F.normalize(k[1:,0,:], dim=-1)\n\n    cosine_qk = (q_cls * k_patch).sum(-1) \n    cosine_qk_max = cosine_qk.max(dim=-1, keepdim=True)[0]\n    cosine_qk_min = cosine_qk.min(dim=-1, keepdim=True)[0]\n    cosine_qk = (cosine_qk-cosine_qk_min) / (cosine_qk_max-cosine_qk_min)\n    return cosine_qk\n\ndef grad_eclip(c, qs, ks, vs, attn_outputs, map_size):\n    ## gradient on last attention output\n    tmp_maps = []\n    for q, k, v, attn_output in zip(qs, ks, vs, attn_outputs):\n        grad = torch.autograd.grad(\n            c,\n            attn_output,\n            retain_graph=True)[0]\n\n        grad_cls = grad[:1,0,:]\n        v_patch = v[1:,0,:]\n        cosine_qk = sim_qk(q, k).reshape(-1)\n        tmp_maps.append((grad_cls * v_patch * cosine_qk[:,None]).sum(-1)) \n\n    emap = F.relu_(torch.stack(tmp_maps, dim=0)).sum(0)\n    return emap.reshape(*map_size)'}
AUTHOR_SHA256 = {'finer_object_prompts.py': '4834a17db74b9a716865954a98b78fd3dafcaebc2434f1ad78fd2a8336d9fd77', 'finer_object_helpers.py': '37cbe633b68f425edfc8c1f94243064689c77c740a46e6d6dee34bd68622eb0f', 'gradeclip_author_functions.py': '41d72e9c60c771955be9460b50132c3bfe928e68033e68cbe3bc8b01bcec6838'}
AUTHOR_REVISIONS = {
    'FINER': 'd67831696e122d01c9823114835f7f50fa07b44c',
    'Grad-ECLIP': 'e370e6cb194faf2020f5d1ed268f9d57e91a38e6',
    'OpenAI-CLIP': 'd05afc436d78f1c48dc0dbf8e5980a9d471f35f6',
    'Transformers': 'v4.57.1'}

def verify_author_sources():
    for name, src in AUTHOR_SOURCES.items():
        if hashlib.sha256(src.encode()).hexdigest() != AUTHOR_SHA256[name]:
            raise RuntimeError(f'Author source modified: {name}')
        ast.parse(src)

verify_author_sources()
FINER_NS = {'List': List}
exec(compile(AUTHOR_SOURCES['finer_object_prompts.py'], '<finer-object-prompts>', 'exec'), FINER_NS)
exec(compile(AUTHOR_SOURCES['finer_object_helpers.py'], '<finer-object-utils>', 'exec'), FINER_NS)

In [ ]:
# These instructions belong to CARA, not FINER/Grad-ECLIP/Qwen paper prompts.
CAPTION_PROMPT = ('Describe the visible image in one concise English sentence, including the main '
                  'objects and their visible attributes. State only what is visible. Output only the caption.')
EVAL_CAPTION_PROMPT = 'Describe this image in one concise English sentence.'
CONCEPT_QUERY_PREFIX = 'Identify visual evidence in the image for this object phrase:\n'

class QwenCaptioner:
    def __init__(self, model_id, revision, device, attention=False):
        from transformers import AutoProcessor, AutoModelForImageTextToText
        self.device = torch.device(device)
        self.processor = AutoProcessor.from_pretrained(model_id, revision=revision,
                            min_pixels=4*32*32, max_pixels=QWEN_MAX_PIXELS)
        with torch.cuda.device(self.device):
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        self.model = AutoModelForImageTextToText.from_pretrained(model_id, revision=revision,
            torch_dtype=dtype, attn_implementation='eager' if attention else 'sdpa').to(self.device)
        self.model.eval().requires_grad_(False)

    def batch(self, messages, image=None):
        prompt = self.processor.apply_chat_template(messages, tokenize=False,
                    add_generation_prompt=True, enable_thinking=False)
        kwargs = {'text': [prompt], 'return_tensors': 'pt'}
        if image is not None: kwargs['images'] = [image]
        return self.processor(**kwargs).to(self.device)

    @torch.inference_mode()
    def generate(self, messages, image=None, sample=False, max_new_tokens=96):
        batch = self.batch(messages,image)
        opts = dict(max_new_tokens=max_new_tokens, do_sample=sample, use_cache=True)
        if sample: opts.update(temperature=CAPTION_TEMPERATURE, top_p=CAPTION_TOP_P)
        ids = self.model.generate(**batch, **opts)
        result = self.processor.batch_decode(ids[:,batch['input_ids'].shape[1]:],
                      skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
        if not result: raise ValueError('VLM returned an empty response.')
        return result

    def caption(self, image, sample=False, evaluation=False):
        messages = [{'role':'user','content':[{'type':'image'},
            {'type':'text','text': EVAL_CAPTION_PROMPT if evaluation else CAPTION_PROMPT}]}]
        return self.generate(messages,image,sample,CAPTION_MAX_NEW_TOKENS)

    def object_record(self, caption):
        if len(self.processor.tokenizer.encode(caption, add_special_tokens=False)) > 512:
            raise ValueError('Caption exceeds FINER 512-token input contract.')
        messages = FINER_NS['_messages_objects'](caption)
        raw = self.generate(messages,max_new_tokens=160)
        if not re.fullmatch(r'PHRASE=[^\n]+',raw) or raw.endswith('.'):
            raise ValueError(f'Invalid FINER response (no fallback): {raw!r}')
        phrase = raw[len('PHRASE='):].strip()
        objects = FINER_NS['_split_objects_phrase'](phrase)
        if not 1 <= len(objects) <= 5 or any(not x.strip() for x in objects):
            raise ValueError(f'Invalid FINER object list: {raw!r}')
        return {'raw_response':raw,'phrase':phrase,'objects':objects}

def caption_bank(captioner, image):
    captions, attempts = [], []
    for _ in range(MAX_BANK_ATTEMPTS):
        raw = captioner.caption(image, sample=True)
        cleaned = ' '.join(raw.split()); key = cleaned.casefold().rstrip('. ')
        duplicate = key in {c.casefold().rstrip('. ') for c in captions}
        attempts.append({'raw_response':raw,'accepted':not duplicate})
        if not duplicate: captions.append(cleaned)
        if len(captions)==K: break
    if len(captions)!=K:
        raise RuntimeError(f'Only {len(captions)}/{K} unique captions; increase attempt budget. No duplicate padding.')
    objects = [captioner.object_record(c) for c in captions]
    counts = Counter(x.casefold() for row in objects for x in set(row['objects']))
    selected = sorted(counts, key=lambda x:(-counts[x],x))[:MAX_CONCEPTS]
    if not selected: raise ValueError('No source concepts; inspect FINER responses.')
    return {'captions':captions,'caption_attempts':attempts,'object_records':objects,
            'concept_counts':dict(counts),'selected_concepts':selected}

In [ ]:
class ClipEnsemble:
    """Persistent per-device workers; autograd flows through device copies to source pixels."""
    def __init__(self,revisions):
        from transformers import CLIPModel,AutoTokenizer
        self.device=torch.device(ATTACK_DEVICE)
        self.models,self.tokenizers={},{}
        for name in BACKBONES:
            ident=CLIP_IDS[name]
            model=CLIPModel.from_pretrained(ident,revision=revisions[ident],
                torch_dtype=torch.float32,low_cpu_mem_usage=True,
                attn_implementation=CLIP_ATTN_IMPLEMENTATION)
            self.models[name]=model.eval().requires_grad_(False).to(CLIP_DEVICE_MAP[name])
            self.tokenizers[name]=AutoTokenizer.from_pretrained(ident,revision=revisions[ident])
        self._initialize_workers()

    def _initialize_workers(self):
        grouped={}
        for name in BACKBONES:
            device=str(next(self.models[name].parameters()).device)
            grouped.setdefault(device,[]).append(name)
        self.groups=[(d,tuple(names)) for d,names in grouped.items()]
        self.normalizers={d:(torch.tensor([.48145466,.4578275,.40821073],device=d)[None,:,None,None],
                              torch.tensor([.26862954,.26130258,.27577711],device=d)[None,:,None,None])
                          for d in grouped}
        self.pool=ThreadPoolExecutor(max_workers=2,thread_name_prefix='cara-clip')
        self.closed=False

    def _collect(self,futures):
        # Drain all jobs before propagating an exception or allowing model cleanup.
        wait(futures)
        outputs={}
        for future in futures: outputs.update(future.result())
        return [outputs[name].to(self.device,non_blocking=True) for name in BACKBONES]

    def _image_group(self,device,names,compact,grad_enabled,ready):
        # grad mode and CUDA current-device selection are thread-local.
        context=torch.cuda.device(device) if torch.device(device).type=='cuda' else contextlib.nullcontext()
        with context,torch.set_grad_enabled(grad_enabled):
            if ready is not None: torch.cuda.current_stream(device).wait_event(ready)
            x=compact.to(device,non_blocking=True)
            mean,std=self.normalizers[device]
            pixels=(x-mean)/std
            result={}
            for name in names:
                offload=grad_enabled and (SAVED_TENSORS_ON_CPU or name in OFFLOAD_BACKBONES)
                memory=(torch.autograd.graph.save_on_cpu(pin_memory=True)
                        if offload and torch.device(device).type=='cuda' else contextlib.nullcontext())
                with memory:
                    result[name]=unit(self.models[name].get_image_features(pixel_values=pixels))
            return result

    def images(self,image):
        if self.closed: raise RuntimeError('CLIP worker pool is closed.')
        # Resize once, retaining the resize/clamp graph on the attack device.
        compact=F.interpolate(image.to(self.device),size=(224,224),mode='bicubic',
                              align_corners=False,antialias=True).clamp(0,1)
        ready=None
        if compact.device.type=='cuda':
            ready=torch.cuda.Event()
            ready.record(torch.cuda.current_stream(compact.device))
        enabled=torch.is_grad_enabled()
        futures=[self.pool.submit(self._image_group,d,names,compact,enabled,ready) for d,names in self.groups]
        return self._collect(futures)

    def _text_group(self,device,names,texts):
        context=torch.cuda.device(device) if torch.device(device).type=='cuda' else contextlib.nullcontext()
        with context,torch.no_grad():
            result={}
            for name in names:
                model=self.models[name]
                tokens=self.tokenizers[name](texts,padding=True,truncation=False,return_tensors='pt')
                if tokens.input_ids.shape[1]>model.config.text_config.max_position_embeddings:
                    raise ValueError(f'{name}: caption exceeds CLIP token limit; no silent truncation.')
                result[name]=unit(model.get_text_features(**tokens.to(device)))
            return result

    @torch.no_grad()
    def texts(self,texts):
        if self.closed: raise RuntimeError('CLIP worker pool is closed.')
        return self._collect([self.pool.submit(self._text_group,d,names,texts) for d,names in self.groups])

    def close(self):
        if getattr(self,'closed',True): return
        self.pool.shutdown(wait=True)
        self.models.clear();self.tokenizers.clear();self.normalizers.clear()
        self.closed=True

    def query(self, image):
        return unit(torch.cat(self.images(image),dim=-1))

    @torch.no_grad()
    def anchor_bank(self, image, captions):
        return fuse_features(self.images(image), self.texts(captions),
            [IMAGE_WEIGHTS[n] for n in BACKBONES], [TEXT_WEIGHTS[n] for n in BACKBONES])

    @torch.no_grad()
    def anchors(self, source, target, banks):
        target_bank = self.anchor_bank(target,banks['target']['captions'])
        source_bank = self.anchor_bank(source,banks['source']['captions'])
        j = nearest_mean_index(target_bank)
        return target_bank[j:j+1].detach(),source_bank.detach(),{
            'target_index':j, 'target_caption':banks['target']['captions'][j],
            'target_distance_to_mean':((target_bank-target_bank.mean(0))**2).sum(-1).sqrt().cpu().tolist(),
            'source_pairwise_cosine':(source_bank@source_bank.T).cpu().tolist(),
            'target_bank':target_bank.cpu().tolist(),'source_bank':source_bank.cpu().tolist()}

In [ ]:
class GradEclipXAI:
    def __init__(self, device):
        import clip
        self.clip = clip
        # Original OpenAI CLIP FP16 GPU model; separate from the HF ensemble.
        model, _ = clip.load('ViT-B/16', device=device, jit=False)
        model.eval().requires_grad_(False)
        self.model, self.device = model,torch.device(device)
        self.ns = dict(torch=torch,F=F,clipmodel=model,
            clip_inres=int(model.visual.input_resolution),clip_ksize=model.visual.conv1.kernel_size)
        exec(compile(AUTHOR_SOURCES['gradeclip_author_functions.py'],'<gradeclip-author>','exec'),self.ns)

    def maps(self, image, concepts):
        # Author imgprocess recipe: resize to nearest patch multiple while preserving aspect.
        image = reduced_image(image)
        w,h = image.size; ph,pw = self.ns['clip_ksize']
        nw,nh = max(pw,int(w/pw+.5)*pw),max(ph,int(h/ph+.5)*ph)
        preprocess = Compose([Resize((nh,nw), interpolation=InterpolationMode.BICUBIC),ToTensor(),
            Normalize((.48145466,.4578275,.40821073),(.26862954,.26130258,.27577711))])
        x = preprocess(image).unsqueeze(0).to(self.device).requires_grad_(True)
        with torch.no_grad():
            text = unit(self.model.encode_text(self.clip.tokenize(concepts,truncate=False).to(self.device)))
        with torch.enable_grad(), contextlib.redirect_stdout(io.StringIO()):
            outputs,_,vs,qs,ks,_,attention_outputs,grid = self.ns['clip_encode_dense'](x,n=1)
            scores = (unit(outputs[:,0])@text.T)[0]
            maps = [self.ns['grad_eclip'](s,qs,ks,vs,attention_outputs,grid).detach().float().cpu()
                    for s in scores]
        return maps,{'grid':list(grid),'model':'OpenAI ViT-B/16','n':1,
                     'author_forward_cosine':scores.detach().float().cpu().tolist()}

def changed_token_span(base_ids, full_ids):
    a,b = base_ids.tolist(),full_ids.tolist(); start=0
    while start<min(len(a),len(b)) and a[start]==b[start]: start+=1
    ea,eb=len(a),len(b)
    while ea>start and eb>start and a[ea-1]==b[eb-1]: ea-=1; eb-=1
    return list(range(start,eb))

class QwenAttentionXAI:
    def __init__(self, captioner): self.captioner=captioner

    @torch.inference_mode()
    def maps(self,image,concepts):
        cap=self.captioner; model=cap.model
        def batch(phrase):
            return cap.batch([{'role':'user','content':[{'type':'image'},
                {'type':'text','text':CONCEPT_QUERY_PREFIX+phrase}]}],image)
        base=batch(''); maps=[]; records=[]
        layers=model.model.language_model.layers
        index=QWEN_ATTENTION_LAYER if QWEN_ATTENTION_LAYER>=0 else len(layers)+QWEN_ATTENTION_LAYER
        if not 0<=index<len(layers): raise ValueError('Qwen attention layer out of range.')
        for phrase in concepts:
            b=batch(phrase); ids=b['input_ids'][0]
            positions=(ids==model.config.image_token_id).nonzero().flatten()
            query=changed_token_span(base['input_ids'][0],ids)
            if not query or not len(positions) or min(query)<=int(positions.max()):
                raise ValueError('Concept token alignment failed.')
            decoded=cap.processor.tokenizer.decode(ids[query],skip_special_tokens=True)
            if re.sub(r'\s+','',phrase).casefold() not in re.sub(r'\s+','',decoded).casefold():
                raise ValueError(f'Unreliable concept span: {phrase!r} -> {decoded!r}')
            t,gh,gw=map(int,b['image_grid_thw'][0].tolist())
            merge=model.config.vision_config.spatial_merge_size
            if t!=1 or gh%merge or gw%merge or len(positions)!=(gh//merge)*(gw//merge):
                raise ValueError('Qwen image token/grid mismatch; expected one static image.')
            captured=[]
            def hook(module,args,output):
                weights=output[1]
                if weights is None: raise RuntimeError('Qwen XAI requires eager attention weights.')
                # Official weights already include causal mask, QK norm, RoPE and all-key softmax.
                selected=weights[0].index_select(1,torch.tensor(query,device=weights.device))
                selected=selected.index_select(2,positions.to(weights.device)).float()
                captured.append(selected.mean((0,1)).detach().cpu())
            handle=layers[index].self_attn.register_forward_hook(hook)
            try:
                # model.model avoids allocating the [sequence,vocabulary] LM logits.
                model.model(**b,use_cache=False,output_attentions=False,return_dict=True)
            finally: handle.remove()
            if len(captured)!=1: raise RuntimeError('Expected exactly one selected attention forward.')
            m=captured[0].reshape(gh//merge,gw//merge); maps.append(m)
            records.append({'concept':phrase,'query_positions':query,'decoded_query':decoded,
                'image_probability_mass':float(m.sum()),'layer':index,'grid':list(m.shape)})
        return maps,{'per_concept':records,'aggregation':'mean heads and concept-token rows'}

In [ ]:
def refresh_xai(xai, adversarial, concepts, step, folder):
    maps,meta=xai.maps(tensor_image(adversarial),concepts)
    valid=[]; errors=[]; arrays={}
    for i,(phrase,m) in enumerate(zip(concepts,maps)):
        arrays[f'concept_{i}']=m.numpy()
        if not torch.isfinite(m).all() or (m<0).any() or m.sum()<=0:
            errors.append({'concept':phrase,'reason':'nonfinite/negative/zero author map'})
            continue
        # Equal weight per valid concept, preserving spatial distribution within each map.
        m=m/m.sum()
        valid.append(F.interpolate(m[None,None],size=(32,32),mode='bilinear',align_corners=False)[0,0])
    if not valid: raise RuntimeError(f'No valid {XAI_METHOD} concept maps at step {step}: {errors}')
    mean=torch.stack([m/m.sum() for m in valid]).mean(0)
    H,W=adversarial.shape[-2:]
    full=F.interpolate(mean[None,None],size=(H,W),mode='bilinear',align_corners=False)[0,0]
    record={'step':step,'method':XAI_METHOD,'concepts':concepts,'excluded_maps':errors,
            'dispersion':dispersion(mean),'metadata':meta}
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    arrays['aggregate_32']=mean.numpy()
    np.savez_compressed(folder/f'xai_{step:03d}.npz',**arrays)
    atomic_json(folder/f'xai_{step:03d}.json',record)
    return full,record

def attack_one(ensemble,xai,source,target,banks,folder,pair_seed):
    seed_everything(pair_seed)
    source=source.to(ATTACK_DEVICE);target=target.to(ATTACK_DEVICE)
    positive,negatives,anchor_record=ensemble.anchors(source,target,banks)
    atomic_json(folder/'anchors.json',anchor_record)
    concepts=banks['source']['selected_concepts']
    adv=source.clone();generator=torch.Generator(device='cpu').manual_seed(pair_seed)
    heat,record=refresh_xai(xai,adv,concepts,0,folder/'xai')
    xai_log=[record];history=[];cached_xai_box=None
    for step in range(1,STEPS+1):
        area=crop_area(step,STEPS,REFRESH_EVERY)
        mode=crop_mode(step,RANDOM_FROM_STEP)
        if mode=="xai":
            if cached_xai_box is None: cached_xai_box=select_box(heat,area,mode,generator)
            box=cached_xai_box
        else: box=select_box(heat,area,mode,generator)
        y,x,h,w=box
        adv=adv.detach().requires_grad_(True)
        # Worker-local saved-tensor contexts are controlled inside ClipEnsemble.
        q=ensemble.query(adv[:,:,y:y+h,x:x+w])
        loss=infonce(q,positive,negatives,TEMPERATURE)
        if not torch.isfinite(loss): raise FloatingPointError(f'Nonfinite loss at {step}.')
        gradient,=torch.autograd.grad(loss,adv)
        gradient_l1=float(gradient.abs().sum())
        if gradient_l1==0: raise RuntimeError(f'Zero attack gradient at step {step}.')
        adv=masked_pgd_step(adv,source,gradient,box,ALPHA_255/255,EPSILON_255/255)
        history.append({'step':step,'loss_before_update':float(loss.detach()),'box_yxhw':list(box),
            'scheduled_area':area,'actual_area':h*w/(source.shape[-2]*source.shape[-1]),
            'crop_mode':mode,'gradient_l1':gradient_l1})
        del q,loss,gradient
        if step%REFRESH_EVERY==0:
            heat,record=refresh_xai(xai,adv,concepts,step,folder/'xai');xai_log.append(record)
            cached_xai_box=None
            atomic_json(folder/'progress.json',{'completed_updates':step,'history':history,'xai':xai_log})
        if step%LOG_EVERY==0: print(f'  update {step}/{STEPS}, {mode}, area={area:.1f}, loss={history[-1]["loss_before_update"]:.4f}')
    with torch.no_grad():
        qs,qa=ensemble.query(source),ensemble.query(adv)
        scores={'clean_global_infonce':float(infonce(qs,positive,negatives,TEMPERATURE)),
            'adversarial_global_infonce':float(infonce(qa,positive,negatives,TEMPERATURE)),
            'clean_target_cosine':float((qs*positive).sum()),
            'adversarial_target_cosine':float((qa*positive).sum()),
            'clean_source_max_cosine':float((qs@negatives.T).max()),
            'adversarial_source_max_cosine':float((qa@negatives.T).max())}
    return adv.detach().cpu(),{'history':history,'xai':xai_log,'surrogate_proxies':scores}

## Kaggle execution and memory

| Stage | GPU0 | GPU1 |
|---|---|---|
| Bank preparation | Qwen3-VL (sequential, deterministic per-pair seeds) | available |
| Attack, Grad-ECLIP | B16+B32 and separate OpenAI CLIP XAI | G14 |
| Attack, Qwen attention | B16+B32 and Qwen3-VL XAI | G14 |
| Caption evaluation | Evaluation VLM; attack models already released | available |
| Text proxies | B16+B32 batched text | G14 batched text |

FP32 is retained for the CLIP attack. Default offload is disabled to avoid copying saved
weights/activations to host memory each update. If memory is insufficient, set
`OFFLOAD_BACKBONES=("G14",)`; this changes the run identity and can be slower.
`SAVED_TENSORS_ON_CPU=True` enables offload for all model branches in their worker threads.
No AMP or silent model substitution is used. There is no assertion of equal GPU utilization.

Persistent workers explicitly inherit grad/no-grad mode, select the CUDA device, and wait
for the resized crop's producer event. Feature `.to(cuda:0)` copies preserve autograd.
Crop ranking is cached within an unchanged XAI stage; random locations still resample
every update. GPU caches are cleared at stage transitions, not after every sample.

These are execution choices, not techniques attributed to the CARA source papers.
References: [PyTorch CUDA semantics](https://docs.pytorch.org/docs/2.8/notes/cuda.html),
[HF CLIP v4.57.1 API](https://huggingface.co/docs/transformers/v4.57.1/model_doc/clip).

Completed samples are hash-verified and skipped on rerun; partial samples restart with
their per-pair seed. Kaggle working storage is session-local: save/download the output
directory or restore it under the same path for later resume. v6.1 has its own run identity.

In [ ]:
def run_identity(revisions):
    names=['SEED','K','STEPS','REFRESH_EVERY','RANDOM_FROM_STEP','EPSILON_255','ALPHA_255',
        'TEMPERATURE','BACKBONES','CLIP_IDS','CLIP_DEVICE_MAP','ATTACK_DEVICE','QWEN_DEVICE','QWEN_ID',
        'IMAGE_WEIGHTS','TEXT_WEIGHTS','CAPTION_TEMPERATURE','CAPTION_TOP_P','CAPTION_MAX_NEW_TOKENS',
        'MAX_BANK_ATTEMPTS','MAX_CONCEPTS','QWEN_ATTENTION_LAYER','QWEN_MAX_PIXELS','XAI_MAX_SIDE',
        'SAVED_TENSORS_ON_CPU','OFFLOAD_BACKBONES','CLIP_ATTN_IMPLEMENTATION','CPU_THREADS',
        'CAPTION_PROMPT','CONCEPT_QUERY_PREFIX','XAI_METHOD']
    config={n:globals()[n] for n in names}
    config.update(model_revisions={k:v for k,v in revisions.items() if k!=EVAL_MODEL_ID},
        author_sources=AUTHOR_SHA256, notebook_code_sha256=NOTEBOOK_CODE_SHA256,
        live_code_sha256=live_code_digest(),
        runtime={p:package_metadata.version(p) for p in ('torch','torchvision','transformers','numpy','Pillow')},
        cuda=torch.version.cuda, hardware=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
    return config

def live_code_digest():
    # Ignore notebook cell filenames/line numbers, but invalidate caches on live code edits.
    import types, inspect
    def constant(value):
        if isinstance(value,types.CodeType): return describe(value)
        if isinstance(value,(tuple,list)): return [constant(c) for c in value]
        if isinstance(value,frozenset): return sorted([constant(c) for c in value],key=repr)
        if isinstance(value,bytes): return {'bytes':value.hex()}
        return repr(value)
    def describe(code):
        return {'bytecode':code.co_code.hex(),'names':list(code.co_names),
            'argcount':code.co_argcount,'kwonly':code.co_kwonlyargcount,
            'consts':[constant(c) for c in code.co_consts]}
    result={}
    namespace=globals()
    for name in sorted(namespace):
        value=namespace[name]
        if isinstance(value,types.FunctionType) and value.__globals__ is namespace:
            result[name]=describe(inspect.unwrap(value).__code__)
        elif isinstance(value,type) and value.__module__==__name__:
            for key,method in vars(value).items():
                if isinstance(method,types.FunctionType): result[name+'.'+key]=describe(inspect.unwrap(method).__code__)
    return digest(result)

def bank_for_pair(pair,captioner,run_dir,identity):
    folder=run_dir/'banks';folder.mkdir(exist_ok=True)
    path=folder/f'{pair["index"]:05d}.json'
    bank_identity={'pair_index':pair['index'],'inputs':pair['input_sha256'],'run':digest(identity)}
    if path.exists():
        record=json.loads(path.read_text(encoding='utf-8'))
        payload={k:v for k,v in record.items() if k!='record_sha256'}
        if record['identity']!=bank_identity or digest(payload)!=record.get('record_sha256'):
            raise RuntimeError(f'Caption bank identity/hash mismatch: {path}')
        if any(len(record[r]['captions'])!=K for r in ('source','target')): raise ValueError('Invalid cached bank K.')
        return record
    record={'identity':bank_identity}
    for j,role in enumerate(('source','target')):
        seed_everything(SEED+pair['index']*1000+j)
        record[role]=caption_bank(captioner,open_rgb(pair[role]))
    record['record_sha256']=digest(record)
    atomic_json(path,record)
    return record

def sample_complete(folder,expected):
    path=folder/'complete.json'
    if not path.exists(): return False
    record=json.loads(path.read_text(encoding='utf-8'))
    if record['identity']!=expected: raise RuntimeError(f'Completed sample identity mismatch: {folder}')
    for filename,sha in record['files'].items():
        p=folder/filename
        if not p.is_file() or file_hash(p)!=sha: raise RuntimeError(f'Artifact corrupted/missing: {p}')
    return True

def export_sample(folder,pair,source,target,adv,record,identity):
    delta=(adv-source).abs()
    if not torch.isfinite(adv).all() or float(delta.max())>EPSILON_255/255+1e-6:
        raise AssertionError('Adversarial tensor exceeds pixel constraint.')
    for role,x in [('source',source),('target',target),('adversarial',adv)]:
        tensor_image(x).save(folder/f'{role}.png')
    saved_source=np.asarray(Image.open(folder/'source.png')).astype(np.int16)
    saved_adv=np.asarray(Image.open(folder/'adversarial.png')).astype(np.int16)
    saved_linf=int(np.abs(saved_adv-saved_source).max())
    if saved_linf>EPSILON_255: raise AssertionError('Saved PNG violates epsilon.')
    record.update(index=pair['index'],inputs=pair,identity=identity,
        tensor_linf_255=float(delta.max()*255),saved_linf_255=saved_linf,
        source_size=list(source.shape[-2:]),target_size=list(target.shape[-2:]))
    atomic_json(folder/'attack.json',record)
    # Complete marker written last; partial samples are restarted from their deterministic seed.
    files={str(p.relative_to(folder)).replace('\\','/'):file_hash(p)
           for p in folder.rglob('*') if p.is_file() and p.name!='complete.json'}
    atomic_json(folder/'complete.json',{'identity':identity,'files':files})

def export_manifest(run_dir,pairs):
    rows=[]
    for pair in pairs:
        folder=run_dir/'samples'/f'{pair["index"]:05d}'
        if not (folder/'complete.json').exists(): continue
        row={'index':pair['index'],'sample_id':f'{pair["index"]:05d}',
            'images':{r:f'samples/{pair["index"]:05d}/{r}.png' for r in ('source','target','adversarial')},
            'original_paths':{r:pair[r] for r in ('source','target')},
            'caption_bank':f'banks/{pair["index"]:05d}.json'}
        rows.append(row)
    (run_dir/f'manifest_{RUN_MODE}_{SAMPLE_START}.jsonl').write_text(
        ''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows),encoding='utf-8')
    return rows

## Đánh giá sau attack

Hai mức đánh giá được giữ riêng:

- **Kiểm tra kỹ thuật:** đủ K/K caption, nearest-mean index, loss/gradient hữu hạn, đúng schedule,
  map đủ 11 mốc, pixel epsilon trên tensor **và PNG**, kích thước source giữ nguyên.
- **Surrogate proxies:** global InfoNCE trước/sau, cosine target/source, khoảng cách caption
  của VLM đánh giá tới banks. Các chỉ số này không phải ground-truth targeted ASR.

VLM đánh giá mặc định Qwen2.5-VL, khác model sinh bank/XAI nhưng vẫn cùng họ Qwen.
VLM chỉ caption ảnh clean/adv; **không nhận target caption** để tránh gợi ý target trong prompt.
Target/source reference vẫn là bank đóng băng. CLIP tính text similarity trên caption của VLM đánh giá
là proxy thuộc surrogate; không khẳng định đây là đánh giá độc lập hoàn toàn.

File `human_review_<mode>_<start>_<evaluator_id>.csv` để reviewer điền `target_success` và
`clean_already_target` (0/1). Báo cáo ASR chỉ xuất khi **tất cả** mẫu được gán nhãn;
báo cả tỷ lệ trên toàn bộ N và tỷ lệ có điều kiện trên clean chưa giống target. Nhãn không được
tự sinh bằng threshold CLIP. Đối chiếu caption/ảnh trực tiếp ở preview trước khi chạy full.
`target_success=1` khi **caption adversarial do VLM đánh giá sinh ra** mô tả đúng chủ thể/object
đích theo target image và bank; `clean_already_target=1` khi caption clean đã đáp ứng cùng tiêu chí.
Không gán thành công chỉ vì caption khác source hoặc chứa một từ trùng target. Ghi trường hợp
mơ hồ vào `review_notes` và thống nhất rubric trước full run. Mỗi evaluator protocol có bảng riêng;
đổi evaluator không reuse nhãn của evaluator cũ. Nếu tắt caption evaluation, chỉ xuất bảng ảnh
để kiểm tra kỹ thuật, không báo VLM ASR.

In [ ]:
def evaluate_caption_outputs(run_dir,rows,revisions):
    eval_config={'model':EVAL_MODEL_ID,'revision':revisions[EVAL_MODEL_ID],
        'prompt':EVAL_CAPTION_PROMPT,'max_new_tokens':CAPTION_MAX_NEW_TOKENS,
        'do_sample':False,'max_pixels':QWEN_MAX_PIXELS,'transformers':package_metadata.version('transformers'),
        'proxy_batch_pairs':EVAL_TEXT_BATCH_PAIRS,'proxy_backend':CLIP_ATTN_IMPLEMENTATION,
        'proxy_code_sha256':NOTEBOOK_CODE_SHA256}
    eval_dir=run_dir/'evaluation'/digest(eval_config)[:16];eval_dir.mkdir(parents=True,exist_ok=True)
    atomic_json(eval_dir/'protocol.json',eval_config)
    evaluator=None;outputs=[]
    try:
        for row in rows:
            path=eval_dir/f'{row["index"]:05d}.json'
            inputs={r:file_hash(run_dir/row['images'][r]) for r in ('source','adversarial')}
            expected={'inputs':inputs,'protocol':digest(eval_config)}
            if path.exists():
                record=json.loads(path.read_text(encoding='utf-8'))
                if record['identity']!=expected: raise RuntimeError('Evaluation cache identity mismatch.')
                payload={k:v for k,v in record.items() if k!='record_sha256'}
                if record.get('record_sha256')!=digest(payload): raise RuntimeError('Evaluation response cache corrupted.')
            else:
                if evaluator is None: evaluator=QwenCaptioner(EVAL_MODEL_ID,revisions[EVAL_MODEL_ID],EVAL_DEVICE)
                seed_everything(SEED+row['index'])
                captions={r:evaluator.caption(open_rgb(run_dir/row['images'][r]),evaluation=True)
                          for r in ('source','adversarial')}
                record={'identity':expected,'index':row['index'],'captions':captions}
                record['record_sha256']=digest(record);atomic_json(path,record)
            outputs.append(record)
    finally:
        del evaluator;free_cuda()
    # VLM is now released. Batch caption text and use both CLIP device groups.
    scores={r['index']:{} for r in rows}
    lookup={o['index']:o for o in outputs}
    ensemble=None
    try:
        ensemble=ClipEnsemble(revisions)
        for offset in range(0,len(rows),EVAL_TEXT_BATCH_PAIRS):
            selected=rows[offset:offset+EVAL_TEXT_BATCH_PAIRS]
            all_texts=[]
            for row in selected:
                bank=json.loads((run_dir/row['caption_bank']).read_text(encoding='utf-8'))
                caps=lookup[row['index']]['captions']
                all_texts += [caps['source'],caps['adversarial']]+bank['target']['captions']+bank['source']['captions']
            with torch.no_grad():
                features=ensemble.texts(all_texts)
                width=2+2*K
                for slot,row in enumerate(selected):
                    for name,feature in zip(BACKBONES,features):
                        emb=feature[slot*width:(slot+1)*width]
                        tgt=unit(emb[2:2+K].mean(0,keepdim=True));src=unit(emb[2+K:].mean(0,keepdim=True))
                        similarities=emb[:2]@torch.cat([tgt,src]).T
                        scores[row['index']][name]={'clean_target':float(similarities[0,0]),
                            'adv_target':float(similarities[1,0]),'clean_source':float(similarities[0,1]),
                            'adv_source':float(similarities[1,1])}
    finally:
        if ensemble is not None: ensemble.close()
        del ensemble;free_cuda()
    results=[]
    for row in rows:
        idx=row['index'];record={'index':idx,'evaluation_protocol':digest(eval_config),**lookup[idx]['captions']}
        for key in ('clean_target','adv_target','clean_source','adv_source'):
            record[key+'_text_cosine_proxy']=float(np.mean([scores[idx][m][key] for m in BACKBONES]))
        record['target_text_cosine_gain_proxy']=record['adv_target_text_cosine_proxy']-record['clean_target_text_cosine_proxy']
        results.append(record)
    pd.DataFrame(results).to_csv(eval_dir/f'caption_proxies_{RUN_MODE}_{SAMPLE_START}.csv',index=False)
    atomic_json(eval_dir/f'raw_scores_{RUN_MODE}_{SAMPLE_START}.json',scores)
    return eval_dir,results

def human_review_table(run_dir,rows,caption_records):
    protocols={r['evaluation_protocol'] for r in caption_records}
    if len(protocols)>1: raise ValueError('Mixed evaluation protocols.')
    protocol=next(iter(protocols)) if protocols else 'image_only_no_vlm'
    path=run_dir/f'human_review_{RUN_MODE}_{SAMPLE_START}_{protocol[:16]}.csv'
    if path.exists():
        frame=pd.read_csv(path)
        if frame['index'].tolist()!=[r['index'] for r in rows]: raise ValueError('Review table indices mismatch.')
        return path
    lookup={r['index']:r for r in caption_records}
    records=[]
    for row in rows:
        bank=json.loads((run_dir/row['caption_bank']).read_text(encoding='utf-8'))
        record={'index':row['index'],'evaluation_protocol':protocol,**{r:str(run_dir/p) for r,p in row['images'].items()},
            'source_bank':json.dumps(bank['source']['captions'],ensure_ascii=False),
            'target_bank':json.dumps(bank['target']['captions'],ensure_ascii=False),
            'clean_vlm_caption':lookup.get(row['index'],{}).get('source',''),
            'adv_vlm_caption':lookup.get(row['index'],{}).get('adversarial',''),
            'target_success':'','clean_already_target':'','review_notes':''}
        records.append(record)
    pd.DataFrame(records).to_csv(path,index=False)
    return path

def report_human_asr(review_csv):
    frame=pd.read_csv(review_csv)
    if 'evaluation_protocol' in frame and (frame['evaluation_protocol']=='image_only_no_vlm').any():
        return {'status':'no_vlm_caption_evaluation','n':len(frame)}
    for col in ('target_success','clean_already_target'):
        if frame[col].isna().any():
            return {'status':'pending_human_labels','n':len(frame),'labeled':int(frame['target_success'].notna().sum())}
        if not frame[col].isin([0,1]).all(): raise ValueError(f'{col} must contain only 0/1.')
    eligible=frame[frame['clean_already_target']==0]
    return {'status':'human_labeled','n':len(frame),'targeted_asr_all':float(frame['target_success'].mean()),
        'n_clean_not_target':len(eligible),
        'targeted_asr_clean_not_target':float(eligible['target_success'].mean()) if len(eligible) else None}

In [ ]:
def self_checks():
    verify_author_sources()
    q=torch.tensor([[.3,.9]],requires_grad=True);p=torch.tensor([[1.,0.]])
    n=torch.tensor([[0.,1.],[-1.,0.],[.1,.9],[-.5,.5],[-.2,-.8]])
    loss=infonce(q,p,n,.17)
    logits=unit(q)@unit(torch.cat([p,n])).T/.17
    torch.testing.assert_close(loss,(torch.logsumexp(logits,-1)-logits[:,0]).mean())
    g,=torch.autograd.grad(loss,q)
    assert torch.isfinite(g).all() and infonce(q-.01*g,p,n,.17)<loss
    assert [crop_area(s) for s in range(1,301,30)]==[.3,.4,.5,.6,.7,.76,.82,.88,.94,1.]
    assert crop_area(150)==.7 and crop_area(151)==.76 and crop_area(300)==1.
    assert sample_count()==NUM_SAMPLES
    assert crop_mode(149)=='xai' and crop_mode(150)=='random'
    assert nearest_mean_index(unit(torch.tensor([[1.,0.],[.8,.2],[-1.,0.]])))==1
    original=torch.full((1,3,10,20),.5);box=(2,4,3,5)
    out=masked_pgd_step(original,original,torch.ones_like(original),box,1/255,16/255)
    assert float((out-original).abs().max())<=16/255+1e-6
    assert torch.equal(out[:,:,:2],original[:,:,:2])
    assert abs(dispersion(torch.ones(4,5))['normalized_entropy']-1)<1e-5
    generator=torch.Generator().manual_seed(1)
    assert select_box(torch.ones(13,21),1.,'random',generator)==(0,0,13,21)
    return {'status':'passed','checks':['author_sha256','InfoNCE_formula_and_descent',
        'nearest_mean_member','schedule_switch_150','full_canvas_final','masked_projection','entropy']}

SELF_CHECK_REPORT=self_checks()
print(SELF_CHECK_REPORT)

In [ ]:
# Resolve and display pair ordering before pretrained inference. No changes to image dimensions.
PAIRS=[];RUN_DIR=None;MANIFEST_ROWS=[];CAPTION_EVALUATION=[]
if RUN_EXPERIMENT:
    validate_config()
    PAIRS=prepare_pairs()
    from IPython.display import display
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(min(5,len(PAIRS)),2,figsize=(10,3*min(5,len(PAIRS))),squeeze=False)
    for row,axes_row in zip(PAIRS,axes):
        for role,ax in zip(('source','target'),axes_row):
            image=open_rgb(row[role]);ax.imshow(image);ax.axis('off')
            ax.set_title(f'{row["index"]} {role}: {image.size}')
    plt.tight_layout();plt.show()
    display(pd.DataFrame(PAIRS)[['index','source','target']])

In [ ]:
def execute_experiment(pairs):
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
    revisions=resolve_revisions();identity=run_identity(revisions)
    run_dir=OUTPUT_ROOT/(XAI_METHOD+'_'+digest(identity)[:16]);run_dir.mkdir(parents=True,exist_ok=True)
    atomic_json(run_dir/'config.json',identity)
    atomic_json(run_dir/'self_checks.json',SELF_CHECK_REPORT)
    print('Run directory:',run_dir)
    captioner=None;ensemble=None;xai=None;probe_done=False
    try:
        # Stage 1: bank creation. Reuse banks only if each identity/hash matches.
        if any(not (run_dir/'banks'/f'{p["index"]:05d}.json').exists() for p in pairs):
            captioner=QwenCaptioner(QWEN_ID,revisions[QWEN_ID],QWEN_DEVICE,
                                    attention=(XAI_METHOD=='qwen_attention'))
        banks={p['index']:bank_for_pair(p,captioner,run_dir,identity) for p in pairs}
        if XAI_METHOD=='grad_eclip':
            del captioner;captioner=None;free_cuda()
        pending=[]
        for pair in pairs:
            folder=run_dir/'samples'/f'{pair["index"]:05d}';folder.mkdir(parents=True,exist_ok=True)
            expected={'run':digest(identity),'inputs':pair['input_sha256'],
                'bank':banks[pair['index']]['record_sha256'],'index':pair['index']}
            if not sample_complete(folder,expected): pending.append((pair,folder,expected))
        if pending:
            if XAI_METHOD=='qwen_attention':
                if captioner is None:
                    captioner=QwenCaptioner(QWEN_ID,revisions[QWEN_ID],QWEN_DEVICE,attention=True)
                xai=QwenAttentionXAI(captioner)
            else: xai=GradEclipXAI(QWEN_DEVICE)
            ensemble=ClipEnsemble(revisions)
            for pair,folder,expected in pending:
                print('Attacking',pair['index'],XAI_METHOD)
                src=image_tensor(open_rgb(pair['source']));tgt=image_tensor(open_rgb(pair['target']))
                if RUN_TWO_GPU_SMOKE_TEST and not probe_done:
                    atomic_json(run_dir/'two_gpu_gradient_probe.json',two_gpu_gradient_probe(ensemble,src))
                    probe_done=True
                for device in (0,1): torch.cuda.reset_peak_memory_stats(device)
                started=time.perf_counter()
                adv,record=attack_one(ensemble,xai,src,tgt,banks[pair['index']],folder,SEED+pair['index'])
                record['gpu_profile']=gpu_profile()
                record['attack_seconds']=time.perf_counter()-started
                record['updates_per_second']=STEPS/record['attack_seconds']
                export_sample(folder,pair,src,tgt,adv,record,expected)
                export_manifest(run_dir,pairs)  # retain a usable partial manifest after interruption
                # Retain allocator pools between samples; clear only at model-stage boundaries.
                del adv,src,tgt
    finally:
        if ensemble is not None: ensemble.close()
        del ensemble,xai,captioner;free_cuda()
    rows=export_manifest(run_dir,pairs)
    if len(rows)!=len(pairs): raise RuntimeError('Incomplete attack manifest.')
    evaluation=[]
    if EVALUATE_CAPTIONS:
        eval_dir,evaluation=evaluate_caption_outputs(run_dir,rows,revisions)
        print('Caption evaluation:',eval_dir)
    review=human_review_table(run_dir,rows,evaluation)
    atomic_json(review.with_suffix('.asr.json'),report_human_asr(review))
    print('Review table:',review)
    return run_dir,rows,evaluation

if RUN_EXPERIMENT:
    RUN_DIR,MANIFEST_ROWS,CAPTION_EVALUATION=execute_experiment(PAIRS)

In [ ]:
# Artifact checks and review; also useful after an interrupted session by setting RUN_DIR manually.
if RUN_DIR is not None:
    import matplotlib.pyplot as plt
    from IPython.display import display
    summary=[]
    for row in MANIFEST_ROWS:
        folder=RUN_DIR/'samples'/f'{row["index"]:05d}'
        attack=json.loads((folder/'attack.json').read_text(encoding='utf-8'))
        bank=json.loads((RUN_DIR/row['caption_bank']).read_text(encoding='utf-8'))
        assert [r['step'] for r in attack['xai']]==list(range(0,301,30))
        assert [r['crop_mode'] for r in attack['history']]==[crop_mode(i) for i in range(1,301)]
        assert attack['history'][-1]['box_yxhw']==[0,0,*attack['source_size']]
        assert attack['saved_linf_255']<=EPSILON_255
        summary.append({'index':row['index'],'saved_linf_255':attack['saved_linf_255'],
                        **attack['surrogate_proxies']})
    display(pd.DataFrame(summary))
    if CAPTION_EVALUATION: display(pd.DataFrame(CAPTION_EVALUATION))
    if MANIFEST_ROWS:
        row=MANIFEST_ROWS[0];folder=RUN_DIR/'samples'/f'{row["index"]:05d}'
        bank=json.loads((RUN_DIR/row['caption_bank']).read_text(encoding='utf-8'))
        anchors=json.loads((folder/'anchors.json').read_text(encoding='utf-8'))
        print('Source captions:',bank['source']['captions'])
        print('Target captions:',bank['target']['captions'])
        print('Selected target index/caption:',anchors['target_index'],anchors['target_caption'])
        print('Source concepts used for XAI:',bank['source']['selected_concepts'])
        fig,axes=plt.subplots(1,4,figsize=(16,4))
        for ax,role in zip(axes[:3],('source','target','adversarial')):
            ax.imshow(open_rgb(folder/f'{role}.png'));ax.set_title(role);ax.axis('off')
        s=np.asarray(open_rgb(folder/'source.png')).astype(float)
        a=np.asarray(open_rgb(folder/'adversarial.png')).astype(float)
        axes[3].imshow(np.abs(a-s).max(-1),vmin=0,vmax=EPSILON_255,cmap='magma')
        axes[3].set_title('max |delta| per pixel (0..epsilon)');axes[3].axis('off')
        plt.tight_layout();plt.show()
        attack=json.loads((folder/'attack.json').read_text(encoding='utf-8'))
        fig,axes=plt.subplots(1,4,figsize=(16,4))
        for ax,step in zip(axes[:3],(0,150,300)):
            heat=np.load(folder/'xai'/f'xai_{step:03d}.npz')['aggregate_32']
            ax.imshow(heat,cmap='inferno');ax.set_title(f'XAI step {step}');ax.axis('off')
        axes[3].plot([r['step'] for r in attack['xai']],
                     [r['dispersion']['normalized_entropy'] for r in attack['xai']],marker='o')
        axes[3].axvline(150,color='gray',linestyle='--');axes[3].set_ylim(0,1.05)
        axes[3].set(xlabel='completed updates',ylabel='normalized XAI entropy')
        plt.tight_layout();plt.show()
    print(f'Processed {len(MANIFEST_ROWS)}/{NUM_SAMPLES} pairs. To scale: set NUM_SAMPLES=200 or 1000; RUN_MODE="full" labels artifacts. Then Run all.')

## Diễn giải kết quả và giới hạn

- Entropy/effective support/spatial variance đo độ phân tán của **map XAI dương** sau khi
  chuẩn hóa về 32×32, không phải raw attention entropy trên toàn bộ sequence và không phải
  entropy của signed gradients. Qwen còn lưu tổng probability mass lên image tokens.
- Bank/objects được freeze từ ảnh clean. Map được refresh trên ảnh adversarial hiện tại.
  Sau step 150, tiếp tục đo map để theo dõi dispersion; crop chọn random theo lịch.
- Gradient trên crop không bảo đảm global InfoNCE hay targeted ASR giảm/tăng đơn điệu.
  Full-frame loss trước/sau được đo riêng. Không suy ra hypothesis đúng chỉ từ một pilot.
- `NUM_SAMPLES` chọn tổng số cặp từ `SAMPLE_START` (mặc định 1; có thể đổi 200 hoặc 1000).
  `RUN_MODE` chỉ đặt nhãn artifacts, không quyết định số mẫu. Mẫu hoàn tất được reuse nếu
  cùng protocol và hash. Chạy cả hai XAI methods tạo hai run directories khác nhau.
- Hai XAI methods hiện sinh bank ở các run riêng và dùng SDPA/eager khác nhau khi captioning.
  Dù cùng seed, đây chưa phải ablation chỉ thay XAI. Muốn so sánh trực tiếp, cần đối chiếu
  toàn bộ bank captions/concepts và selected target; không kết luận do XAI nếu bank khác nhau.
- Nguồn prompt Grad-ECLIP/FINER được ghim và kiểm tra SHA trong notebook; các link nguồn
  đầu notebook trỏ tới đúng revision đã dùng.
- Chưa benchmark pretrained pipeline, VRAM hay ASR trong môi trường tác giả notebook này.
  Test trước 1 mẫu để kiểm tra pipeline, rồi tăng `NUM_SAMPLES` khi đã review kết quả.

- v6.1 is optimized structurally for two GPUs, but throughput/VRAM must be measured on Kaggle. 